In [1]:
Sys.setenv(TZ = "America/Vancouver")

In [2]:
install.packages("suncalc")

Installing package into 'C:/Users/Kaiyan Zhang/AppData/Local/R/win-library/4.4'
(as 'lib' is unspecified)



package 'suncalc' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\Kaiyan Zhang\AppData\Local\Temp\RtmpYTql7D\downloaded_packages


In [3]:
library(sf)
library(dplyr)
library(lubridate)
library(suncalc)
library(readr)
library(purrr)


Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.3.1; sf_use_s2() is TRUE

Warning message:
"package 'dplyr' was built under R version 4.4.2"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union


Warning message:
"package 'suncalc' was built under R version 4.4.3"


In [4]:
# divisions <- st_read("census_divison.geojson", quiet = TRUE)
# divisions_valid <- st_make_valid(divisions)
# sf_use_s2(FALSE)
# vancouver_union <- st_union(divisions_valid)
# st_write(vancouver_union, "vancouver_union.geojson", delete_dsn = TRUE)


In [5]:
Sys.setenv(TZ = "America/Vancouver")

vancouver_neighborhood <- st_read("../Boundary_data/local-area-boundary.geojson", quiet = TRUE)

census_division <- st_read("../Boundary_data/census_divison.geojson", quiet = TRUE)
census_division <- st_make_valid(census_division)


crime_data_all <- read_csv("../crimedata_csv_AllNeighbourhoods_AllYears/crimedata_csv_AllNeighbourhoods_AllYears.csv")


Rows: 910710 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): TYPE, HUNDRED_BLOCK, NEIGHBOURHOOD
dbl (7): YEAR, MONTH, DAY, HOUR, MINUTE, X, Y

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [6]:

crime_data_clean <- crime_data_all |>
  filter(!is.na(X), !is.na(Y)) |>
  filter((X != 0.0) & (Y != 0.0))|>
  filter(!TYPE %in% c("Vehicle Collision or Pedestrian Struck (with Fatality)", "Vehicle Collision or Pedestrian Struck (with Injury)"))


In [7]:

crime_sf <- st_as_sf(crime_data_clean, coords = c("X", "Y"), crs = 26910)

crime_sf <- st_transform(crime_sf, 4326)


In [8]:

crime_sf <- st_join(crime_sf, census_division["name"])|>
  glimpse()


Rows: 811,219
Columns: 10
$ TYPE          <chr> "Break and Enter Commercial", "Break and Enter Commercia…
$ YEAR          <dbl> 2012, 2019, 2019, 2021, 2014, 2020, 2021, 2022, 2005, 20…
$ MONTH         <dbl> 12, 3, 8, 4, 8, 7, 11, 1, 11, 5, 7, 6, 4, 9, 1, 11, 2, 4…
$ DAY           <dbl> 14, 7, 27, 26, 8, 28, 21, 5, 14, 21, 1, 1, 17, 1, 3, 14,…
$ HOUR          <dbl> 8, 2, 4, 4, 5, 19, 6, 7, 3, 4, 0, 20, 5, 14, 16, 20, 18,…
$ MINUTE        <dbl> 52, 6, 12, 44, 13, 12, 33, 34, 9, 50, 35, 0, 50, 20, 19,…
$ HUNDRED_BLOCK <chr> NA, "10XX SITKA SQ", "10XX ALBERNI ST", "10XX ALBERNI ST…
$ NEIGHBOURHOOD <chr> "Oakridge", "Fairview", "West End", "West End", "West En…
$ geometry      <POINT [°]> POINT (-123.1197 49.23361), POINT (-123.129 49.266…
$ name          <chr> "59150544", "59153689", "59153483", "59153483", "5915348…


In [9]:

crime_sf$crime_datetime <- as.POSIXct(
  paste(crime_sf$YEAR,
        crime_sf$MONTH,
        crime_sf$DAY,
        crime_sf$HOUR,
        crime_sf$MINUTE,
        sep = "-"),
  format = "%Y-%m-%d-%H-%M",
  tz = "America/Vancouver"
)

crime_sf$date_only <- as.Date(crime_sf$crime_datetime)

unique_dates <- unique(crime_sf$date_only)
sun_times <- getSunlightTimes(
  date = unique_dates,
  lat = 49.2827,
  lon = -123.1207,
  keep = c("sunrise", "sunset"),
  tz = "America/Vancouver"
) %>%
  select(date, sunrise, sunset)

crime_sf <- left_join(crime_sf, sun_times, 
                                by = c("date_only" = "date"))

crime_sf$time_category <- ifelse(
  crime_sf$crime_datetime >= crime_sf$sunrise &
    crime_sf$crime_datetime < crime_sf$sunset,
  "day",
  "night"
)


In [10]:
names(crime_sf) <- tolower(names(crime_sf))
crime_sf <- crime_sf|>
    select(!c(crime_datetime, date_only, sunrise, sunset, minute))|>
    filter(!is.na(time_category))|>
    st_drop_geometry()

glimpse(crime_sf)

Rows: 811,181
Columns: 9
$ type          <chr> "Break and Enter Commercial", "Break and Enter Commercia…
$ year          <dbl> 2012, 2019, 2019, 2021, 2014, 2020, 2021, 2022, 2005, 20…
$ month         <dbl> 12, 3, 8, 4, 8, 7, 11, 1, 11, 5, 7, 6, 4, 9, 1, 11, 2, 4…
$ day           <dbl> 14, 7, 27, 26, 8, 28, 21, 5, 14, 21, 1, 1, 17, 1, 3, 14,…
$ hour          <dbl> 8, 2, 4, 4, 5, 19, 6, 7, 3, 4, 0, 20, 5, 14, 16, 20, 18,…
$ hundred_block <chr> NA, "10XX SITKA SQ", "10XX ALBERNI ST", "10XX ALBERNI ST…
$ neighbourhood <chr> "Oakridge", "Fairview", "West End", "West End", "West En…
$ name          <chr> "59150544", "59153689", "59153483", "59153483", "5915348…
$ time_category <chr> "day", "night", "night", "night", "night", "night", "nig…


In [11]:

# crime_sf <- st_set_geometry(crime_sf, NULL)

write_csv(crime_sf, "crime_in_vancouver_census.csv")

crime_day <- crime_sf %>% filter(time_category == "day")
write_csv(crime_day, "crime_day_census.csv")

crime_night <- crime_sf %>% filter(time_category == "night")
write_csv(crime_night, "crime_night_census.csv")

cat("处理完成：\n",
    "1) crime_in_vancouver_with_division.csv\n",
    "2) crime_day.csv\n",
    "3) crime_night.csv\n")


处理完成：
 1) crime_in_vancouver_with_division.csv
 2) crime_day.csv
 3) crime_night.csv
